# 01 — MLL23 dataset audit

Reads the checked-in M1 artifacts (manifest, audit report, split manifest, leakage
report) and visualises class distribution and split fractions. Run from the
`notebooks/` directory; all paths below are relative to the repository root.

Artifacts must already exist — see `bloodfilm dataset audit`, `build-manifest` and
`dataset split`. This notebook never downloads or rebuilds anything.

In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path

REPO_ROOT = Path("..").resolve()
MANIFEST = REPO_ROOT / "data" / "manifests" / "mll23_manifest.csv"
AUDIT_REPORT = REPO_ROOT / "outputs" / "reports" / "mll23_audit.json"
SPLITS = REPO_ROOT / "data" / "manifests" / "mll23_splits.csv"
LEAKAGE_REPORT = REPO_ROOT / "outputs" / "reports" / "mll23_leakage.json"
for path in (MANIFEST, AUDIT_REPORT, SPLITS, LEAKAGE_REPORT):
    assert path.exists(), f"missing M1 artifact: {path}"

## Class distribution

Log scale: MLL23 spans two orders of magnitude (reactive lymphocytes vs.
segmented neutrophils). Rare-class support is reported raw alongside every metric.

In [ ]:
import matplotlib.pyplot as plt

with MANIFEST.open(newline="", encoding="utf-8") as handle:
    labels = [row["canonical_label"] for row in csv.DictReader(handle)]
counts = Counter(labels)
print(f"valid images: {len(labels)}")
for name, count in counts.most_common():
    print(f"{name:28s} {count:6d}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(counts.keys(), counts.values())
ax.set_yscale("log")
ax.set_ylabel("images (log scale)")
ax.set_title("MLL23 class distribution")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
plt.show()

## Audit: dimensions and unreadable files

In [ ]:
audit = json.loads(AUDIT_REPORT.read_text(encoding="utf-8"))
print("dimensions:", audit["dimensions"])
print("total files:", audit["total_files"])
print("unreadable:", len(audit["unreadable"]))
for path in audit["unreadable"][:10]:
    print(" ", path)

## Split fractions and leakage

Groups are surrogate per-image IDs until official MLL23 patient metadata is
available, so a clean leakage report is expected but claims nothing about true
patient independence (`independence_claim` must stay false).

In [ ]:
with SPLITS.open(newline="", encoding="utf-8") as handle:
    split_counts = Counter(row["split"] for row in csv.DictReader(handle))
total = sum(split_counts.values())
for split, count in split_counts.most_common():
    print(f"{split:12s} {count:6d} ({count / total:.1%})")
print(json.dumps(json.loads(LEAKAGE_REPORT.read_text(encoding="utf-8")), indent=2))